In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:90%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# 백터 DB : Chroma vs Pinecone
- Chroma : 인메모리DB, 로컬메모리DB
- Pinecone : 클라우드 vector DB
    - (Pinecone console에 api key 생성 → .env(PINECONE_API_KEY 등록))

# 1. Knowledge Base 구성을 위한 데이터 생성

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200)

document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
len(document_list)

In [3]:
# embedding : upstage embedding-query
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
    # model="embedding-query"
)

In [4]:
%%time

# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 저장할 때 / 업로드 할 때
index_name = "tax-index-upstage"
database = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=embeddings,
    index_name=index_name
)

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: total: 13.5 s
Wall time: 1min 14s


In [ ]:
# %%time
# # 업로드한 백터DB 가져오기
# from pinecone import Pinecone
# from langchain_pinecone import PineconeVectorStore
# index_name = "tax-index"

# database = PineconeVectorStore(
#     embedding=embedding,         # 질물을 임베딩하여 유사도 검색
#     index_name=index_name
# )

# 2. 제공되는 prompt를 활용하여 답변 생성

In [5]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

In [8]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [9]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(), # database.as_retriever()
    chain_type_kwargs={"prompt":prompt}
)

In [10]:
ai_message = qa_chain.invoke({'query':query})
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 근로소득공제(최대 2천만원 공제 후 계산)와 기타 공제 조건에 따라 달라질 수 있지만, 구체적인 세액 계산은 제공된 정보만으로는 알 수 없습니다. 소득세는 과세표준과 세율에 따라 결정되며, 이 경우에는 근로소득공제 후 금액이 중요합니다. 따라서 정확한 세액을 계산하려면 상세한 연말 소득세 신고 과정이 필요합니다.'}